# Using Claude to generate the Verhaal Speciaal story

** Code has been tested with Python 3.12 **

In this notebook we will create a 'Verhaal Speciaal' story.

The story will be generated by a LLM (GPT-4) based on a prompt. 

The story will be based on user input. One of the user inputs is the reading level, which is based on class/group. 

In the last part of the notebook we can evaluate the level of the generated text.

### Contents
0. Installs and imports
1. Settings and prompt
2. Generate chapter one
3. Generate chapter two & three
4. Convert story to JSON

## 0. Installs and imports

In [121]:
#!pip install openai --upgrade

In [122]:
import anthropic
anthropic.__version__

'0.51.0'

In [123]:
#import the local files
import config
import leesniveaus

## 1. Settings

### Setting the reading levels
Four different reading levels have been defined, see below.

Both characters have their own reading level as Verhaal Speciaal is meant to be a reading combination for parent and child. Example: parent can have reading level 3, while the child can have reading level 1.  All combinations are possible. 

### User input

The user_input prompt collects the input the user of the story creates. This is taken from the javascript code of the original Verhaal Speciaal:
1. personage een        => character_one
2. personage twee       => character_two
3. wat                  => plot
4. waarom               => reasoning
5. waar                 => setting
6. wanneer              => time

And we set the reading levels:
7. klas/groep           => groep (will be mapped to reading_level)
8. Leesniveau ouder     => reading level

### Building the prompt 

The reading levels and user input are then used to build up a prompt..

We will generate a prompt consisting of three 'sub-prompts':

prompt =  basic_prompt + previous_text + chapter_prompt 

**basic_prompt**

The basic prompt sets the structure of the story. It defines there are two characters and a story teller.  It makes sure the story follows a pattern.  

**previous_text**

Only used for chapters 2 and 3. 
Input of the previous chapter(s): one or two. 

**chapter_prompt**

This are the chapter specific inputs:

1. Start new story, introduce characters, leave room for chapters 2 and 3
2. Follow up on chapter 1, use previous text and leave room for chapter 3
3. chapter 3: Final chapter, end the story, use previous text.

### Reading levels

In [124]:
#variables based on the reading level settings
level_one = leesniveaus.level_one
level_two = leesniveaus.level_two
level_three = leesniveaus.level_three
level_four = leesniveaus.level_four


### User input

In [125]:
#These are the variables from the front end about the story . 
character_one = 'eddy'
character_two = 'jan'
plot ='een wandeling'
reasoning = 'ze verdwalen'
setting = 'in het bos' #waar
time = 'in de zomer'

# These are the input variables from the front end for the reading level
group_child = 7 #class the child is in 3,4,5,6,7,8
reading_level_parent = level_four # 1 2 3 4 based on reading level settings 

In [126]:
#Reading level conversion table CHILD

group = group_child #class the child is in 3,4,5,6,7,8

if group < 4:
    reading_level_child = level_one
    print(reading_level_child)
elif group == 4:
    reading_level_child = level_two
    print(reading_level_child)
elif group <= 6:
    reading_level_child = level_three
    print(reading_level_child)
elif group <= 8:
    reading_level_child = level_four
    print(reading_level_child)


Leesniveau 4: 
Samengestelde zinnen komen voor. 
Lastige leenwoorden zijn ook toegestaan.
Speciale leestekens  (ideeën, ruïne, saté, Curaçao) komen meer voor. 
Woorden eindigend op -ele, -eaal, -ueel, -iaal of -ieel komen voor. 
Ook woorden beginnend met /ch/ uitgesproken als /sj/, eindigend op –ge, uitgesproken als /zje/, eindigend op –isch, woorden met klinkerreeks, leenwoorden met eau, é of è. 
Hoofdletters worden gebruikt.



In [127]:
#summarizing the reading levels
print(f'Reading level child: {reading_level_child[:12]}')
print(f'Reading level parent: {reading_level_parent[:12]}')

Reading level child: Leesniveau 4
Reading level parent: Leesniveau 4


### basic_prompt

In [128]:
#update reading levels
basic_prompt_v4 =f'''Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is {reading_level_parent}.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is een beschrijving van personage {character_one}.
Het leesniveau van personage {character_one} is niveau {reading_level_child}, dus houd het taalgebruik op dat niveau voor dit personage. Gebruik hiervoor de omschrijving van de hiervoor genoemde niveuas
Dit is een beschrijving van personage {character_two}.
Het leesniveau van personage {character_two} is niveau {reading_level_parent}, dus houd het taalgebruik op dat niveau voor dit personage. Gebruik hiervoor de omschrijving van de hiervoor genoemde niveaus houdt het hoofdstuk bij twee zinnen per karakter.

De algemene verhaallijn is: {plot}.
Dit is de reden achter het verhaal: {reasoning}.
De setting van het verhaal is: {setting}.
De tijd waarin het verhaal zich afspeelt is: {time}.

Gebruik de volgende regels om te output te structureren:
Iedere zin of paragraaf van het verhaal moet bij de Verteller, {character_one} of {character_two} horen. 
De verteller wordt altijd aangeduid als Verteller. Gebruik het format Verteller | tekst
Voeg geen code tussen haakjes toe voor de Verteller.

Als een personage wat gaat vertellen, voeg {{char1}} of {{char2}} toe voor de naam van het personage dat spreekt.
voeg een | tussen alle woorden in zoals in dit voorbeeld: {{char1}} | {character_one} | tekst.
Aan het einde van het hoofdstuk moet de verteller een vraag stellen aan een van de personages over de voorgaande dialoog.
Aan het einde van het hoofdstuk moet de tekst '''"{ENDOFACT}"''' op een nieuwe regel worden toegevoegd.
Begin het hoofdstuk duidelijk met het nummer van het hoofdstuk. Bijvoorbeeld: 'Hoofdstuk 1'.
Zorg ervoor dat de personages hetzelfde blijven in de verschillende hoofdstukken en dat ze weten wat er gezegd is.
Voeg geen uitleg toe, alleen de dialoog.
Voeg geen nieuwe personages of settings toe aan de dialoog.
De allereerste regel van de tekst moet een gegenereerde titel zijn. Gebruik alleen letters en spaties, in de titel staat niet het woord 'titel'.
Gebruik geen speciale tekens in de tekst, alleen letters, spaties en nieuwe regels.
'''

### Previous text

In [129]:
#for chapter one empty, for chapter 2/3 will be updated, see below
previous = " " 

### Chapter prompts

In [130]:
chapter_1 = f'''Dit is het eerste hoofdstuk van drie, zorg dus dat het verhaal verder kan gaan.'''
chapter_2 = f'''Dit is het tweede hoofdstuk van drie. Ga door op het eerste hoofdstuk wat je uit deze tekst haalt: {previous}. Zorg dat het verhaal verder kan gaan in hoofdstuk 3. Begin de tekst met de titel van het verhaal en dan Hoofdstuk 2 '''
chapter_3 = f'''Dit is het laatste hoofdstuk dus zorg voor een goed en happy einde. Ga door met het verhaal gebaseerd op hoofdstuk 1 en 2 wat je uit de deze tekst haalt: {previous}. Begin de tekst met de titel van het verhaal en dan Hoofdstuk 3.'''

### Concatenate prompt
We will use v3 as this is the 3rd version of the prompt (to keep it similar to original Verhaal Speciaal)

prompt_v3 = basic_prompt + chapter_prompt

In [131]:
prompt_v4_ch1 = basic_prompt_v4 + chapter_1 
print(prompt_v4_ch1)

Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is Leesniveau 4: 
Samengestelde zinnen komen voor. 
Lastige leenwoorden zijn ook toegestaan.
Speciale leestekens  (ideeën, ruïne, saté, Curaçao) komen meer voor. 
Woorden eindigend op -ele, -eaal, -ueel, -iaal of -ieel komen voor. 
Ook woorden beginnend met /ch/ uitgesproken als /sj/, eindigend op –ge, uitgesproken als /zje/, eindigend op –isch, woorden met klinkerreeks, leenwoorden met eau, é of è. 
Hoofdletters worden gebruikt.
.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is een beschrijving van personage eddy.
Het leesniveau van personage eddy is niveau Leesniveau 4: 
Samengestelde zinnen komen voor. 

## 2. Generate chapter one

In [132]:
import anthropic
import config

client = anthropic.Anthropic(
    # defaults to os.environ.get("ANTHROPIC_API_KEY")
    api_key=config.ANTHROPIC_API_KEY,
)
message = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=1024,
    messages=[
        {"role": "user", "content": prompt_v4_ch1}
    ]
)
print(message.content[0])

TextBlock(citations=None, text='Verdwaald in het Groene Bos\n\nHoofdstuk 1\n\nVerteller | De zomerzon schijnt helder door de groene bladeren van het mysterieuze bos. Eddy en Jan lopen enthousiast over het smalle bospad, maar plotseling realiseren ze zich dat ze de verkeerde route hebben genomen.\n\n{char1} | eddy | Hé Jan, herken jij deze plek nog want ik zie helemaal geen bekende bomen of struiken meer.\n\n{char2} | jan | Nee Eddy, dit pad ziet er totaal anders uit dan vanmorgen en ik denk dat we een verkeerde afslag hebben genomen bij die grote eik.\n\n{char1} | eddy | Misschien moeten we teruglopen naar het begin, want anders worden onze ouders natuurlijk ongerust als we te laat thuiskomen.\n\n{char2} | jan | Dat is een logische gedachte Eddy, maar ik weet niet meer welke richting we vandaan kwamen omdat alle paden er hetzelfde uitzien.\n\nVerteller | Eddy, welk gevoel krijg jij nu je beseft dat jullie verdwaald zijn in dit uitgestrekte bos?\n\n{ENDOFACT}', type='text')


In [ ]:
#Function to call the Anthropic API
def create_chat_completion(prompt, model="claude-sonnet-4-20250514"):
  
    message = client.messages.create(
        model= model,
        max_tokens=1024,
        messages=[
            {"role": "user", 
             "content": prompt}
        ]   
    )

    # Return the generated response
    chapter = message.content[0].text
    return chapter


In [134]:
chapter_one = create_chat_completion(prompt_v4_ch1)
print(chapter_one)

De verdwaling in het groene bos

Hoofdstuk 1

Verteller | De zomerse middag was ideaal voor een wandeling door het mysterieuze bos. Eddy en Jan liepen enthousiast over het smalle bospad, terwijl de zonnestralen door de bladeren dansten.

{char1} | eddy | Wat een fantastische dag om te wandelen, Jan! Het bos ruikt zo fris en natuurlijk.

{char2} | jan | Absoluut waar, Eddy! De vogels zingen melodieus en de bloemen bloeien prachtig langs het pad.

Verteller | Na een tijdje bereikten ze een kruispunt waar verschillende paden in verschillende richtingen leidden. De bomen werden dichter en het werd moeilijker om de juiste weg te herkennen.

{char1} | eddy | Hmm, welke route moeten we kiezen? Dit kruispunt ziet er verwarrend uit met al die opties.

{char2} | jan | Ik denk dat we links moeten gaan, want daar lijkt het pad breder en makkelijker begaanbaar te zijn.

Verteller | Eddy, waarom denk je dat jullie dit specifieke pad gekozen hebben?

{ENDOFACT}


## 3. Generate chapter two & three

The first chapter is input for chapter two and three. 



In [135]:
previous = f"Het vorige hoofdstuk was: {chapter_one}."

In [136]:
prompt_v4_ch2 = basic_prompt_v4+previous+chapter_2
print(prompt_v4_ch2)

Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is Leesniveau 4: 
Samengestelde zinnen komen voor. 
Lastige leenwoorden zijn ook toegestaan.
Speciale leestekens  (ideeën, ruïne, saté, Curaçao) komen meer voor. 
Woorden eindigend op -ele, -eaal, -ueel, -iaal of -ieel komen voor. 
Ook woorden beginnend met /ch/ uitgesproken als /sj/, eindigend op –ge, uitgesproken als /zje/, eindigend op –isch, woorden met klinkerreeks, leenwoorden met eau, é of è. 
Hoofdletters worden gebruikt.
.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is een beschrijving van personage eddy.
Het leesniveau van personage eddy is niveau Leesniveau 4: 
Samengestelde zinnen komen voor. 

In [137]:
chapter_two = create_chat_completion(prompt_v4_ch2)
print(chapter_two)

De verdwaling in het groene bos

Hoofdstuk 2

Verteller | Na een half uur lopen door het dichte bos realiseerden Eddy en Jan zich dat het pad smaller werd en uiteindelijk helemaal verdween. De bomen stonden zo dicht bij elkaar dat ze nauwelijks nog zonlicht konden zien.

{char1} | eddy | Jan, ik geloof dat we de verkeerde richting hebben gekozen en nu compleet verdwaald zijn.

{char2} | jan | Je hebt gelijk, Eddy! Ik herken deze omgeving helemaal niet meer en zie geen bekende oriëntatiepunten.

Verteller | De twee vrienden keken om zich heen en probeerden wanhopig een herkenbaar pad of wegwijzer te vinden. Het bos leek eindeloos en alle richtingen zagen er hetzelfde uit.

{char1} | eddy | Misschien moeten we teruglopen naar het kruispunt en een andere route proberen te vinden.

{char2} | jan | Dat is een uitstekend idee, maar ik weet niet meer precies welke kant we gekomen zijn.

Verteller | Jan, wat denk je dat jullie nu het beste kunnen doen om de weg terug te vinden?

{ENDOFACT}


In [138]:
#calling chapter trhee
previous = f"De vorige hoofdstukken waren {chapter_one} en {chapter_two}."
prompt_v4_ch3 = basic_prompt_v4+previous+chapter_3
print(prompt_v4_ch3)

Je bent een kinderboekenschrijver. 
Je schrijft een verhaal het Nederlands waarbij je de drie-hoofdstukken-structuur van een toneelstuk volgt.
Dit is een scriptdialoog tussen twee personages en er is een Verteller die de scène schetst. 

Het leesniveau van de verteller is Leesniveau 4: 
Samengestelde zinnen komen voor. 
Lastige leenwoorden zijn ook toegestaan.
Speciale leestekens  (ideeën, ruïne, saté, Curaçao) komen meer voor. 
Woorden eindigend op -ele, -eaal, -ueel, -iaal of -ieel komen voor. 
Ook woorden beginnend met /ch/ uitgesproken als /sj/, eindigend op –ge, uitgesproken als /zje/, eindigend op –isch, woorden met klinkerreeks, leenwoorden met eau, é of è. 
Hoofdletters worden gebruikt.
.

Er zijn twee karakers die ieder een eigen leesniveau hebben. Hierna volgen de regels per niveau. Daarna wordt aangegeven welk niveau ieder personage heeft. 
Dit is een beschrijving van personage eddy.
Het leesniveau van personage eddy is niveau Leesniveau 4: 
Samengestelde zinnen komen voor. 

In [139]:
chapter_three = create_chat_completion(prompt_v4_ch3)
print(chapter_three)

De verdwaling in het groene bos

Hoofdstuk 3

Verteller | Plotseling hoorden Eddy en Jan het geluid van stromend water in de verte. Ze liepen voorzichtig in de richting van het geluid en ontdekten een heldere beek die door het bos kronkelde.

{char1} | eddy | Luister Jan, dat water klinkt als muziek en misschien kunnen we de beek volgen naar de uitgang.

{char2} | jan | Briljant idee Eddy, want water stroomt meestal naar beneden en leidt vaak naar bewoonde gebieden.

Verteller | De vrienden volgden de beek gedurtig en na een tijdje zagen ze in de verte de rand van het bos. Hun hart sprong op van vreugde toen ze de bekende weilanden en het dorp herkenden.

{char1} | eddy | Kijk daar Jan, ons dorp ligt precies waar we het verwachtten en we zijn veilig teruggekeerd.

{char2} | jan | Wat een avontuurlijke wandeling was dit, Eddy, en gelukkig hebben we samen de oplossing gevonden.

Verteller | Jan, wat heb je vandaag geleerd over het belang van samenwerking tijdens deze spannende boswandeli

## 4. Save story as JSON

We need to create a .json file out of this story. This .json file will be analysed by the other scripts in the 'AVI Score' repository. 

The JSON structure is relatively simple, containing just one key-value pair. The complexity lies in the structured text content rather than in nested JSON objects or arrays.

The JSON file contains a single object with one key-value pair:
Key: "text"
Value: A long string containing a story

- First line contains the title
- Narrator sections: Paragraphs starting with "Verteller |"
- Character dialogues: Lines starting with "{char1} |" or "{char2} |"

Character dialogues follow this pattern:
- {char1} | anna | [dialogue text]
- {char2} | tom | [dialogue text]

Other things
- "{ENDOFACT}" appears at the end, likely indicating the end of a story act or section.
- The text uses newline characters (\n) to separate lines and sections.
- There are no nested objects or arrays within this JSON structure.
- The entire story is contained within a single string value.


In [140]:
story = {"text": chapter_one + chapter_two + chapter_three}
print(story['text'])

De verdwaling in het groene bos

Hoofdstuk 1

Verteller | De zomerse middag was ideaal voor een wandeling door het mysterieuze bos. Eddy en Jan liepen enthousiast over het smalle bospad, terwijl de zonnestralen door de bladeren dansten.

{char1} | eddy | Wat een fantastische dag om te wandelen, Jan! Het bos ruikt zo fris en natuurlijk.

{char2} | jan | Absoluut waar, Eddy! De vogels zingen melodieus en de bloemen bloeien prachtig langs het pad.

Verteller | Na een tijdje bereikten ze een kruispunt waar verschillende paden in verschillende richtingen leidden. De bomen werden dichter en het werd moeilijker om de juiste weg te herkennen.

{char1} | eddy | Hmm, welke route moeten we kiezen? Dit kruispunt ziet er verwarrend uit met al die opties.

{char2} | jan | Ik denk dat we links moeten gaan, want daar lijkt het pad breder en makkelijker begaanbaar te zijn.

Verteller | Eddy, waarom denk je dat jullie dit specifieke pad gekozen hebben?

{ENDOFACT}De verdwaling in het groene bos

Hoofdst

In [141]:
import json
import os
from datetime import datetime

# Convert the string into a JSON serializable format, e.g., as a dictionary
story_to_save = story

# Get the current date
current_date = datetime.now().strftime("%Y-%m-%d_%H:%M")

# Create a filename with the current date
filename = f"vs_claude_{current_date}.json"
file_path = os.path.join('json', filename)

# Save the data to a JSON file
with open(file_path, 'w') as json_file:
    json.dump(story_to_save, json_file, indent=4)

print(f"Data saved to {filename}")

Data saved to vs_claude_2025-05-27_13:36.json


## 5. Validate the new .json with an old example

In [142]:
#Here we use an old .json based on the Javascript code base

import json
from pprint import pprint

# Read the JSON file
with open('./json/V_S_2025-05-27_12:20.json', 'r') as file:
    data = json.load(file)

# Pretty print using json.dumps()
print("Pretty printed using json.dumps():")
print(json.dumps(data, indent=4))

# Pretty print using pprint
print("\nPretty printed using pprint:")
pprint(data['text'])

Pretty printed using json.dumps():
{
    "text": "Verdwaald in het Groene Bos\n\nHoofdstuk 1\n\nVerteller | De zomerzon schijnt helder door de bladeren van het dichte bos. Eddy en Jan lopen samen over een smal bospad, terwijl de vogels vrolijk fluiten in de bomen boven hun hoofden.\n\n{char1} | eddy | Wat een prachtige dag voor een wandeling door dit mysterieuze bos!\n\n{char2} | jan | Inderdaad, de natuur is hier heel speciaal en de lucht ruikt zo fris.\n\n{char1} | eddy | Kijk eens naar die kronkelende paden die alle kanten opgaan.\n\n{char2} | jan | Welke route moeten we eigenlijk nemen om terug te komen?\n\nVerteller | Eddy, herinner je je nog welke weg jullie genomen hebben om hier te komen?\n\n{ENDOFACT}De Verdwaalde Wandelaars\n\nHoofdstuk 1\n\nVerteller | Het is een prachtige zomerdag en Eddy en Jan beslissen om een lange wandeling te maken door het dichte bos. De zon schijnt helder tussen de bladeren door en overal horen ze vogels zingen.\n\n{char1} | eddy | Wat een fantastisc

## To do list
- take prompts to config files instead of code

 
